# Trial Counts by Stage and Mouse

Summary of total trials, free-choice trials, and session counts across
all training stages (4.2–4.7) and experimental sessions (4.8) for 6 mice.

In [6]:
import sys, os
import numpy as np
import pandas as pd

_d = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isfile(os.path.join(_d, "config.py")):
        break
    _d = os.path.dirname(_d)
sys.path.insert(0, os.path.join(_d, "src"))
sys.path.insert(0, _d)

import data_import as di
from config import SUBJECTS, raw_subject_dir

In [7]:
STAGES = ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8']

records = []

for subject in SUBJECTS:
    # Training sessions (stages 4.2–4.7)
    exp_tr = di.Experiment(raw_subject_dir(subject, training=True))
    for s in exp_tr.get_sessions(subject_IDs='all', when='all'):
        stage = s.training_stage
        if stage in STAGES[:6]:
            lines = s.print_lines
            fc = sum(1 for line in lines if line.split(' ')[9] == 'CT:FC')
            records.append({
                'subject': subject, 'stage': stage,
                'total_trials': len(lines), 'fc_trials': fc
            })

    # Experimental sessions → labeled as stage 4.8
    exp_ex = di.Experiment(raw_subject_dir(subject, training=False))
    for s in exp_ex.get_sessions(subject_IDs='all', when='all'):
        lines = s.print_lines
        fc = sum(1 for line in lines if line.split(' ')[9] == 'CT:FC')
        records.append({
            'subject': subject, 'stage': '4.8',
            'total_trials': len(lines), 'fc_trials': fc
        })

df = pd.DataFrame(records)
print(f"{len(df)} sessions loaded across {df['subject'].nunique()} mice")
print(f"Total trials: {df['total_trials'].sum()}")
df.head(10)

Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
Saved sessions loaded from: sessions.pkl
193 sessions loaded across 6 mice
Total trials: 55974


,subject,stage,total_trials,fc_trials
0,WT1,4.2,95,23
1,WT1,4.3,73,18
2,WT1,4.4,102,26
3,WT1,4.5,103,51
4,WT1,4.6,97,73
5,WT1,4.6,157,117
6,WT1,4.6,215,161
7,WT1,4.6,207,155
8,WT1,4.6,247,185
9,WT1,4.7,229,172


## Total Trials per Stage

In [12]:
# Pooled summary by stage (all sessions pooled, for reference)
stage_summary = df.groupby('stage')['total_trials'].agg(['count', 'sum', 'mean', 'std']).round(1)
stage_summary.columns = ['n_sessions', 'total', 'pooled_mean', 'pooled_std']

print(f"\nGrand total: {df['total_trials'].sum()} trials")

# Per-mouse mean total trials, then mean +/- SD ACROSS the 6 mice.
# This is the aggregation used in the paper table: each mouse contributes one
# value (its session-mean), and the SD is the spread across mice.
# ddof=0 (population SD, divide by n) reproduces the paper table exactly.
pivot_total = df.groupby(['subject', 'stage'])['total_trials'].mean().unstack()[STAGES]
pivot_total.loc['Mean'] = pivot_total.mean()
pivot_total.loc['Std']  = pivot_total.loc[SUBJECTS].std(ddof=0)   # population SD across mice
pivot_total.round(1)


Grand total: 55974 trials


stage,4.2,4.3,4.4,4.5,4.6,4.7,4.8
subject,,,,,,,
WT1,95.0,73.0,102.0,103.0,184.6,256.9,338.2
WT2,84.0,74.0,58.0,60.0,190.5,346.5,428.1
WT3,77.0,100.0,82.0,112.0,166.4,277.3,376.7
WT4,70.0,88.0,97.0,89.0,203.2,325.2,436.6
WT5,62.0,102.0,67.0,108.0,170.6,307.0,364.6
WT6,161.0,91.0,141.0,94.5,264.0,338.7,393.9
Mean,91.5,88.0,91.2,94.4,196.5,308.6,389.7
Std,32.8,11.3,27.1,17.2,32.6,32.3,34.5


## Free-Choice Trials per Stage

In [9]:
# Per-mouse mean FC trials, then mean +/- SD across the 6 mice (ddof=0)
pivot_fc = df.groupby(['subject', 'stage'])['fc_trials'].mean().unstack()[STAGES]
pivot_fc.loc['Mean'] = pivot_fc.mean()
pivot_fc.loc['Std']  = pivot_fc.loc[SUBJECTS].std(ddof=0)
pivot_fc.round(1)

stage,4.2,4.3,4.4,4.5,4.6,4.7,4.8
subject,,,,,,,
WT1,23.0,18.0,26.0,51.0,138.2,192.8,253.7
WT2,22.0,18.0,15.0,30.0,142.8,260.0,321.2
WT3,19.0,25.0,20.0,56.0,124.9,207.7,282.6
WT4,18.0,22.0,25.0,44.0,152.3,243.9,327.5
WT5,15.0,25.0,17.0,54.0,127.8,230.4,273.4
WT6,40.0,23.0,35.0,47.5,198.0,254.1,295.4
Mean,22.8,21.8,23.0,47.1,147.3,231.5,292.3
Std,8.1,2.9,6.7,8.6,24.4,24.3,25.9


In [10]:
# ── Paper-table summary: mean ± SD across mice (per-mouse session means) ──
def stage_mean_std(col, ddof=0):
    pm = df.groupby(['subject', 'stage'])[col].mean().unstack()[STAGES]
    return pm.mean(), pm.std(ddof=ddof)

tot_m, tot_s = stage_mean_std('total_trials')
fc_m,  fc_s  = stage_mean_std('fc_trials')

# sessions per mouse (mean across the mice that reached each stage)
sess = df.groupby(['subject', 'stage']).size().unstack()[STAGES]
sess_per_mouse = sess.mean()

summary = pd.DataFrame({
    'total_trials':     [f"{tot_m[s]:.1f} ± {tot_s[s]:.1f}" for s in STAGES],
    'fc_trials':        [f"{fc_m[s]:.1f} ± {fc_s[s]:.1f}"  for s in STAGES],
    'fc_fraction':      [f"{fc_m[s] / tot_m[s]:.3f}"            for s in STAGES],
    'sessions_per_mouse': [f"{sess_per_mouse[s]:.1f}"          for s in STAGES],
}, index=STAGES)
summary.index.name = 'stage'
print("mean ± SD across 6 mice (per-mouse session means; SD is population, ddof=0)")
summary

mean ± SD across 6 mice (per-mouse session means; SD is population, ddof=0)


,total_trials,fc_trials,fc_fraction,sessions_per_mouse
stage,,,,
4.2,91.5 ± 32.8,22.8 ± 8.1,0.250,1.0
4.3,88.0 ± 11.3,21.8 ± 2.9,0.248,1.0
4.4,91.2 ± 27.1,23.0 ± 6.7,0.252,1.0
4.5,94.4 ± 17.2,47.1 ± 8.6,0.499,1.2
4.6,196.5 ± 32.6,147.3 ± 24.4,0.750,7.3
4.7,308.6 ± 32.3,231.5 ± 24.3,0.750,5.8
4.8,389.7 ± 34.5,292.3 ± 25.9,0.750,14.8


## Session Counts per Stage

In [11]:
# Sessions per mouse per stage
pivot_sessions = df.groupby(['subject', 'stage']).size().unstack(fill_value=0)
pivot_sessions = pivot_sessions[STAGES]
pivot_sessions.loc['Total'] = pivot_sessions.sum()
pivot_sessions

stage,4.2,4.3,4.4,4.5,4.6,4.7,4.8
subject,,,,,,,
WT1,1,1,1,1,5,10,11
WT2,1,1,1,1,11,2,17
WT3,1,1,1,1,11,3,14
WT4,1,1,1,1,6,8,15
WT5,1,1,1,1,9,5,16
WT6,1,1,1,2,2,7,16
Total,6,6,6,7,44,35,89
